# 01 — Strict econometric logit benchmark (Káčer & Alexy style)

A reference logistic regression for Slovak firm failure prediction. The notebook
is positioned as the econometric benchmark against the non-linear and sequence
models (EBM, BiLSTM); it therefore prioritises methodological cleanliness,
coefficient interpretability and replicability over raw discriminative
performance.

Pipeline:

1. **Out-of-time split** — train 2013–2016, validation 2017, test 2018–2019.
2. **Feature set** — liquidity, leverage, capital structure, profitability,
   activity, size and age variables. Pre-computed ratios from notebook 00 are
   reused when present; missing ratio columns can be created from raw accounting
   items, and remaining missing values are handled by training-period median
   imputation after the time split.
3. **Industry fixed effects without the dummy-variable trap** — the most
   frequent industry in the training period is dropped as the reference.
4. **Year fixed effects** — `year_2014`, `year_2015`, `year_2016`; reference
   year is 2013.
5. **VIF filter** applied only to non-protected continuous variables.
   Categorical blocks are diagnosed with the Fox & Monette (1992)
   `GVIF^(1/(2·df))`.
6. **Full-rank filter** — drops variables that are exact linear combinations
   of the constant and previously kept variables (a technical safeguard, not a
   model-selection step).
7. **Logit-1 screening** with sign and significance checks, followed by an
   **iterative Logit-2 refit** that drops the worst remaining wrong-sign or
   non-significant variable until the model is clean.
8. **Diagnostics on the test set** — ROC-AUC, PR-AUC, KS, Brier, balanced
   accuracy, recall and precision at the validation Youden threshold,
   Hosmer–Lemeshow goodness-of-fit, calibration slope/intercept,
   Brier Skill Score, top-k capture and lift, standardised odds ratios, and a
   per-year AUC decomposition (2018 vs 2019).

Inputs: `data_logit_strict.csv` (produced by notebook 00, read from `DATA_DIR` or the current working directory).
Outputs: `logit_v7_results.pkl` (predictions + metadata for the stacking
notebook) and `logit_v7_results.json` (metrics + diagnostics summary).

References: Fox & Monette (1992); O'Brien (2007); Hosmer, Lemeshow & Sturdivant
(2013); Steyerberg et al. (2010); Saito & Rehmsmeier (2015); Brier (1950).

## 1. Imports and configuration

Loads numpy, pandas, statsmodels, sklearn metrics and scipy.stats. Defines `DATA_DIR` from the environment variable `DATA_DIR` when available, otherwise from the current working directory. The notebook reads `data_logit_strict.csv` and writes `logit_v7_results.pkl` plus `logit_v7_results.json`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, json, time, pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (
    roc_curve, roc_auc_score, average_precision_score,
    accuracy_score, brier_score_loss, balanced_accuracy_score,
    confusion_matrix
)
from scipy import stats as scistats

# === Paths ===
# For GitHub/reproducibility: either run the notebook in the folder containing
# the prepared CSV files or set the DATA_DIR environment variable.
DATA_DIR  = os.environ.get("DATA_DIR", os.getcwd())
DATA_PATH = os.path.join(DATA_DIR, "data_logit_strict.csv")
RESULTS_PATH = os.path.join(DATA_DIR, "logit_v7_results.pkl")
JSON_PATH    = os.path.join(DATA_DIR, "logit_v7_results.json")

print(f"DATA_PATH: {DATA_PATH}")
print(f"  existuje: {os.path.exists(DATA_PATH)}")

## 2. Candidate variables and expected signs

`base_features` lists the literature-driven candidate ratios across liquidity, leverage / capital structure, profitability, activity, interest coverage, size, age and non-financials. `EXPECTED_SIGNS` encodes the economic prior for each variable — `+` means an increase predicts higher failure probability, `-` the opposite. These priors are enforced during the Logit-1 screening and the Logit-2 iterative refit.

In [ ]:
base_features = [
    # Likvidita
    'cash_ratio','quick_ratio','current_ratio','cash_ta','nwc_ta',
    # Zadĺženosť a kapitálová štruktúra
    'debt_ta','ltd_ta','cl_ta','eq_ta','eq_tl','bl_ta',
    # Rentabilita
    'roa1','roa2','re_ta','rotc','roltc','ebitda_sales','ros',
    # Aktivita
    'asset_turnover','ta_days','ca_days','rec_days','creditors_days',
    'creditors_days_log','pay_days','pay_days_adj',
    # Krytie úrokov + veľkosť firmy
    'icr','log_ta',
    # Vek + nefinančné
    'age_log','joint_stock','micro_company','small_company','medium_company',
    # Interakcia
    'int_bl_ebitda',
]
year_features = ['year_2014','year_2015','year_2016']

EXPECTED_SIGNS = {
    'cash_ratio':'-','quick_ratio':'-','current_ratio':'-','cash_ta':'-','nwc_ta':'-',
    'debt_ta':'+','ltd_ta':'+','cl_ta':'+','bl_ta':'+',
    'eq_ta':'-','eq_tl':'-',
    'roa1':'-','roa2':'-','re_ta':'-','rotc':'-','roltc':'-','ebitda_sales':'-','ros':'-',
    'icr':'-','asset_turnover':'-',
    'ta_days':'+','ca_days':'+','rec_days':'+','creditors_days':'+',
    'creditors_days_log':'+','pay_days':'+','pay_days_adj':'+',
    'int_bl_ebitda':'-',
    'joint_stock':'+','micro_company':'+','small_company':'+','medium_company':'+',
    'age_log':'+',  # empiricky pozit. pre SR (Káčer & Alexy 2025)
    'log_ta':'-',   # väčšie firmy menej zlyhávajú
}
print(f"Base features: {len(base_features)}")
print(f"Year FE: {year_features}")

## 3. Compute financial ratios

`compute_ratios()` derives engineered ratios from raw accounting items only when they are not already present in the dataset (the notebook 00 outputs are kept untouched). Helpers: `_div` for safe division returning `NaN` on zero denominators, `_get` for fallback column lookup with aliases, and `_set_if_missing` to avoid overwriting pre-computed columns.

In [ ]:
def _div(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    return pd.Series(np.where((den == 0) | den.isna(), np.nan, num/den), index=num.index)

def _set_if_missing(d, col, vals):
    if col not in d.columns:
        d[col] = vals

def _get(d, *names, default=np.nan):
    for n in names:
        if n in d.columns:
            return pd.to_numeric(d[n], errors="coerce")
    return pd.Series(default, index=d.index)

def compute_ratios(df):
    d = df.copy()
    TA   = _get(d, 'celkove_aktiva')
    CA   = _get(d, 'obezne_aktiva')
    INV  = _get(d, 'zasoby', default=0).fillna(0)
    CASH = _get(d, 'peniaze', default=0).fillna(0)
    AR   = _get(d, 'obchodne_pohladavky', default=0).fillna(0)
    EQ   = _get(d, 'vlastne_imanie')
    RE   = _get(d, 'nerozdeleny_zisk', default=0).fillna(0)
    TL   = _get(d, 'celkove_zavazky')
    LTL  = _get(d, 'dlhodobe_zavazky', default=0).fillna(0)
    CL   = _get(d, 'kratkodobe_zavazky')
    AP   = _get(d, 'obchodne_zavazky', default=0).fillna(0)
    BL   = (_get(d, 'bankove_uvery_dlhodobe', default=0).fillna(0)
            + _get(d, 'bankove_uvery_kratkodobe', default=0).fillna(0))
    if 'trzby_spolu' in d.columns:
        REV = _get(d, 'celkove_trzby').fillna(_get(d, 'trzby_spolu', default=np.nan))
    else:
        REV = _get(d, 'celkove_trzby')
    DEP     = _get(d, 'odpisy', default=0).fillna(0)
    EBT     = _get(d, 'EBT', 'ebt')
    INT_EXP = _get(d, 'nakladove_uroky', default=0).fillna(0)
    INT_INC = _get(d, 'vynosove_uroky', default=0).fillna(0)
    NI      = _get(d, 'cisty_zisk_strata')

    NWC    = CA - CL
    EBIT   = EBT + INT_EXP - INT_INC
    EBITDA = EBIT + DEP

    # Likvidita
    _set_if_missing(d, 'cash_ratio',    _div(CASH, CL))
    _set_if_missing(d, 'quick_ratio',   _div(CA-INV, CL))
    _set_if_missing(d, 'current_ratio', _div(CA, CL))
    _set_if_missing(d, 'cash_ta',       _div(CASH, TA))
    _set_if_missing(d, 'nwc_ta',        _div(NWC, TA))
    # Zadĺženosť
    _set_if_missing(d, 'debt_ta',       _div(TL, TA))
    _set_if_missing(d, 'ltd_ta',        _div(LTL, TA))
    _set_if_missing(d, 'cl_ta',         _div(CL, TA))
    _set_if_missing(d, 'eq_ta',         _div(EQ, TA))
    _set_if_missing(d, 'eq_tl',         _div(EQ, TL))
    _set_if_missing(d, 'bl_ta',         _div(BL, TA))
    # Rentabilita
    _set_if_missing(d, 'roa1',          _div(NI, TA))
    _set_if_missing(d, 'roa2',          _div(EBIT, TA))
    _set_if_missing(d, 're_ta',         _div(RE, TA))
    _set_if_missing(d, 'rotc',          _div(NI+DEP, TL))
    _set_if_missing(d, 'roltc',         _div(EBIT, EQ+LTL))
    _set_if_missing(d, 'ebitda_sales',  _div(EBITDA, REV))
    _set_if_missing(d, 'ros',           _div(NI, REV))
    # Aktivita
    _set_if_missing(d, 'asset_turnover',_div(REV, TA))
    _set_if_missing(d, 'ta_days',       _div(TA*365, REV))
    _set_if_missing(d, 'ca_days',       _div(CA*365, REV))
    _set_if_missing(d, 'rec_days',      _div(AR*365, REV))
    _set_if_missing(d, 'creditors_days',_div(AP*365, REV))
    _set_if_missing(d, 'pay_days',      _div(CL*365, REV))
    _set_if_missing(d, 'pay_days_adj',  _div(TL*365, REV))
    # Krytie úrokov + veľkosť
    _set_if_missing(d, 'icr',    _div(EBIT, INT_EXP))
    _set_if_missing(d, 'log_ta', np.log1p(TA.clip(lower=0)))

    if 'creditors_days' in d.columns:
        _set_if_missing(d, 'creditors_days_log',
            np.log1p(pd.to_numeric(d['creditors_days'],errors='coerce').clip(lower=0)))
    if 'bl_ta' in d.columns and 'ebitda_sales' in d.columns:
        _set_if_missing(d, 'int_bl_ebitda', d['bl_ta'] * d['ebitda_sales'])

    if 'pravna_forma' in d.columns:
        _set_if_missing(d, 'joint_stock',
            d['pravna_forma'].astype(str).str.contains('Akciov',case=False,na=False).astype(float))
    if 'pocet_zamestnancov' in d.columns:
        emp = pd.to_numeric(d['pocet_zamestnancov'],errors='coerce')
        _set_if_missing(d, 'micro_company',  ((emp<10) | emp.isna()).astype(float))
        _set_if_missing(d, 'small_company',  ((emp>=10)&(emp<50)).astype(float))
        _set_if_missing(d, 'medium_company', ((emp>=50)&(emp<250)).astype(float))
    if 'age_log' not in d.columns and 'datum_vzniku' in d.columns:
        dt = pd.to_datetime(d['datum_vzniku'], errors='coerce')
        d['age_log'] = np.log1p(
            (pd.to_numeric(d['rok'],errors='coerce') - dt.dt.year).clip(lower=0))
    return d

print("compute_ratios() pripravený.")

## 4. VIF filter — variable-type-aware logic

The classical VIF only has a clean interpretation for continuous variables, so
the notebook splits the candidate set into three groups:

1. **True dummy variables** — `industry_*`, `nace_*`, `year_*`, `joint_stock`,
   `micro_company`, `small_company`, `medium_company`.
2. **Protected continuous / derived variables** — `log_ta` and
   `int_bl_ebitda`, kept regardless of VIF for methodological reasons.
3. **Other continuous variables** — the only group on which the iterative
   VIF filter is applied.

The industry block is additionally diagnosed via `GVIF^(1/(2·df))`
(Fox & Monette, 1992).

In [ ]:
DUMMY_PREFIXES = ('industry_', 'nace_', 'year_')
TRUE_DUMMY_NAMES = {
    'joint_stock', 'micro_company', 'small_company', 'medium_company',
}
PROTECTED_FEATURES = {
    'log_ta', 'int_bl_ebitda',
}

def is_dummy(col):
    """Skutočné kategorické/binárne premenné."""
    return any(col.startswith(p) for p in DUMMY_PREFIXES) or col in TRUE_DUMMY_NAMES

def is_protected(col):
    """Premenné ponechané mimo VIF filtra: dummy bloky + vybrané ekonomicky dôležité spojité premenné."""
    return is_dummy(col) or col in PROTECTED_FEATURES


def _rank_filter_priority(col):
    """
    Poradie pre full-rank filter.
    Premenné s nižšou prioritou sa skúšajú ponechať skôr; ak vznikne presná
    lineárna závislosť, odstráni sa až neskoršia redundantná premenná.
    """
    if col in PROTECTED_FEATURES:
        return (0, col)
    if not is_dummy(col):
        return (1, col)
    if col.startswith('year_'):
        return (2, col)
    if col in TRUE_DUMMY_NAMES:
        return (3, col)
    if col.startswith(('industry_', 'nace_')):
        return (4, col)
    return (5, col)

def _rank_fast(A, tol=None):
    """Rýchla kontrola hodnosti cez Gramovu maticu; vhodné pre veľa riadkov a málo stĺpcov."""
    return np.linalg.matrix_rank(A.T @ A, tol=tol)

def make_full_rank_features(X_train, features, tol=None, verbose=True):
    """
    Odstráni presne redundantné stĺpce vzhľadom na konštantu a už ponechané premenné.

    Dôležité: nejde o ekonomický/modelový výber premenných. Funkcia rieši iba
    technický problém neplnej hodnosti dizajnovej matice pred odhadom Logit-1.
    """
    features = list(dict.fromkeys(features))
    ordered = sorted(features, key=_rank_filter_priority)

    n = len(X_train)
    current = np.ones((n, 1), dtype=float)  # konštanta ostáva vždy vo modeli
    current_rank = _rank_fast(current, tol=tol)

    kept = []
    dropped = []
    for f in ordered:
        col = pd.to_numeric(X_train[f], errors='coerce').fillna(0).to_numpy(dtype=float).reshape(-1, 1)
        candidate = np.column_stack([current, col])
        candidate_rank = _rank_fast(candidate, tol=tol)
        if candidate_rank > current_rank:
            kept.append(f)
            current = candidate
            current_rank = candidate_rank
        else:
            dropped.append({
                'feature': f,
                'reason': 'exact_linear_dependence_with_constant_or_previous_features'
            })

    kept_set = set(kept)
    kept_original_order = [f for f in features if f in kept_set]

    # Kontrola výsledku po návrate k pôvodnému poradiu stĺpcov.
    X_check = sm.add_constant(X_train[kept_original_order].fillna(0), has_constant='add')
    rank_check = _rank_fast(np.asarray(X_check.values, dtype=float), tol=tol)
    if rank_check < X_check.shape[1]:
        raise RuntimeError(
            f"Full-rank filter zlyhal: rank={rank_check}, columns={X_check.shape[1]}. "
            "Skontroluj duplicitné alebo konštantné premenné."
        )

    if verbose:
        print(f"  [full-rank] pred: {len(features)} prem. | po: {len(kept_original_order)} prem. | vyradené: {len(dropped)}")
        if dropped:
            print("  [full-rank] vyradené presne redundantné premenné:", [d['feature'] for d in dropped])
    return kept_original_order, dropped

def compute_vif(X_train, features):
    if not features:
        return pd.DataFrame(columns=['premenná', 'VIF'])
    X = X_train[features].fillna(0)
    vifs = []
    for i in range(len(features)):
        try:
            vifs.append(variance_inflation_factor(X.values, i))
        except Exception:
            vifs.append(np.nan)
    return (pd.DataFrame({'premenná': features, 'VIF': vifs})
              .sort_values('VIF', ascending=False).reset_index(drop=True))

def filter_vif(X_train, features, threshold=10.0, verbose=True):
    """VIF filter sa aplikuje iba na nechránené spojité premenné."""
    continuous_tested = [f for f in features if not is_protected(f)]
    protected = [f for f in features if is_protected(f)]
    if verbose:
        n_dummy = sum(1 for f in protected if is_dummy(f))
        n_prot_cont = sum(1 for f in protected if f in PROTECTED_FEATURES)
        print(f"  [VIF] spojité na test: {len(continuous_tested)} | "
              f"chránené dummy: {n_dummy} | chránené spojité: {n_prot_cont}")
    remaining = continuous_tested.copy()
    removed = []
    while True:
        if len(remaining) < 2:
            break
        X = X_train[remaining].fillna(0)
        vifs = {}
        for i, c in enumerate(remaining):
            try:
                vifs[c] = variance_inflation_factor(X.values, i)
            except Exception:
                vifs[c] = np.nan
        worst = max(vifs, key=lambda k: vifs[k] if not np.isnan(vifs[k]) else 0)
        wv = vifs[worst]
        if np.isnan(wv) or wv <= threshold:
            break
        removed.append(f"{worst} (VIF={wv:.1f})")
        remaining.remove(worst)
    if verbose:
        if removed:
            print(f"  [VIF] odstránené spojité: {removed}")
        else:
            print(f"  [VIF] žiadna testovaná spojitá premenná nepresiahla VIF={threshold}")
    return remaining + protected

def compute_gvif_block(X_train, block_features, other_features):
    """Orientačné GVIF^(1/(2*df)) podľa Fox & Monette (1992) pre kategoriálny blok."""
    if not block_features:
        return None
    X = X_train[block_features + other_features].fillna(0)
    n_block = len(block_features)
    vifs = []
    for i in range(n_block):
        try:
            vifs.append(variance_inflation_factor(X.values, i))
        except Exception:
            vifs.append(np.nan)
    max_vif = float(np.nanmax(vifs)) if vifs else np.nan
    try:
        Rb = X_train[block_features].fillna(0).corr().values
        det_block = float(np.linalg.det(Rb))
        gvif = 1.0 / det_block if det_block and det_block > 0 else np.nan
        gvif_norm = gvif ** (1/(2*n_block)) if gvif and gvif > 0 else np.nan
    except Exception:
        gvif = np.nan
        gvif_norm = np.nan
    return {
        'n_block': n_block,
        'max_vif_within_block': max_vif,
        'GVIF': gvif,
        'GVIF_normalized': gvif_norm,
        'note': 'GVIF_normalized < 3.16 približne zodpovedá klasickému VIF < 10.'
    }

print("VIF + GVIF funkcie pripravené.")

## 5. Fit functions — Logit-1 screening + iterative Logit-2 refit

Wraps `statsmodels.Logit` with safety checks: `_design_matrix` validates full rank before fitting, `_fit_robust` tries Newton / BFGS / L-BFGS / Powell / Nelder-Mead and prefers the first converged result, and `validate_logit_result` aborts the notebook on non-convergence, NaN p-values or infinite standard errors. `select_from_logit1` keeps only variables whose coefficient sign matches the prior and whose p-value is below `alpha`. `fit_logit2_iterative` then drops the single worst remaining wrong-sign or non-significant variable per iteration until the model is clean.

In [ ]:
def _design_matrix(X_train, features, step_name="Logit"):
    X = sm.add_constant(X_train[features].fillna(0), has_constant='add')
    X_np = np.asarray(X.values, dtype=float)
    rank = np.linalg.matrix_rank(X_np.T @ X_np)
    if rank < X.shape[1]:
        raise RuntimeError(
            f"{step_name}: dizajnová matica nie je plnej hodnosti "
            f"(rank={rank}, columns={X.shape[1]}). Skontroluj dummy premenné, referenčné kategórie "
            "a nulovú variabilitu premenných v tréningovej množine."
        )
    return X

def _fit_robust(y, X):
    """Skúsi viac optimalizačných metód a preferuje konvergovaný výsledok."""
    last_error = None
    last_result = None
    for m in ['newton', 'bfgs', 'lbfgs', 'powell', 'nm']:
        try:
            res = sm.Logit(y, X).fit(method=m, maxiter=2000, disp=False)
            if bool(res.mle_retvals.get('converged', False)):
                return res
            last_result = res
        except Exception as e:
            last_error = e
    if last_result is not None:
        return last_result
    raise RuntimeError(f"Logit fit failed: {last_error}")

def validate_logit_result(res, step_name, features=None, max_nan_p_share=0.0, require_converged=True):
    """Zastaví notebook pri nekonvergencii, NaN p-hodnotách alebo nekonečných SE."""
    converged = bool(res.mle_retvals.get('converged', False))
    if require_converged and not converged:
        raise RuntimeError(f"{step_name}: model nekonvergoval. Výber premenných by nebol spoľahlivý.")

    tbl = res.summary2().tables[1]
    checked_features = features if features is not None else [c for c in tbl.index if c != 'const']
    checked_features = [c for c in checked_features if c in tbl.index and c != 'const']

    if checked_features:
        pvals = pd.to_numeric(tbl.loc[checked_features, 'P>|z|'], errors='coerce')
        ses = pd.to_numeric(tbl.loc[checked_features, 'Std.Err.'], errors='coerce')
        nan_p_share = float(pvals.isna().mean())
        if nan_p_share > max_nan_p_share:
            bad = pvals[pvals.isna()].index.tolist()[:10]
            raise RuntimeError(
                f"{step_name}: podiel NaN p-hodnôt = {nan_p_share:.1%}. "
                f"Prvé problematické premenné: {bad}. Skontroluj kolinearitu alebo separáciu."
            )
        if np.isinf(ses).any():
            bad = ses[np.isinf(ses)].index.tolist()[:10]
            raise RuntimeError(
                f"{step_name}: nekonečné štandardné chyby pri premenných {bad}. "
                "Model nie je vhodný na ekonometrickú interpretáciu."
            )
    print(f"  [{step_name}] kontrola OK | converged={converged} | NaN p-values ≤ {max_nan_p_share:.0%}")
    return tbl

def fit_logit1(X_train, y, features):
    X = _design_matrix(X_train, features, step_name='Logit-1')
    return _fit_robust(y, X)

def select_from_logit1(res, features, signs=EXPECTED_SIGNS, alpha=0.05):
    tbl = res.summary2().tables[1]
    keep, dropped = [], []
    for c in features:
        if c not in tbl.index:
            dropped.append((c, 'missing'))
            continue
        p = tbl.loc[c, 'P>|z|']
        b = tbl.loc[c, 'Coef.']
        if pd.isna(p):
            dropped.append((c, 'nan_p'))
            continue
        if p >= alpha:
            dropped.append((c, f'p={p:.3g}'))
            continue
        s = signs.get(c)
        if s is None:
            keep.append(c)
        elif s == '+' and b > 0:
            keep.append(c)
        elif s == '-' and b < 0:
            keep.append(c)
        else:
            dropped.append((c, f'wrong_sign b={b:.3f}'))
    return keep, dropped

def fit_logit2_iterative(X_train, y, init_features, signs=EXPECTED_SIGNS,
                          alpha=0.05, min_features=3):
    features = list(init_features)
    history = []
    while True:
        if len(features) < min_features:
            break
        X = _design_matrix(X_train, features, step_name='Logit-2')
        res = _fit_robust(y, X)
        validate_logit_result(res, 'Logit-2 intermediate', features, max_nan_p_share=0.0, require_converged=True)
        tbl = res.summary2().tables[1]
        wrong, nonsig = [], []
        for c in features:
            if c not in tbl.index:
                continue
            b = tbl.loc[c, 'Coef.']
            p = tbl.loc[c, 'P>|z|']
            s = signs.get(c)
            if s is not None and ((s == '+' and b <= 0) or (s == '-' and b >= 0)):
                wrong.append((c, b, p))
            elif p >= alpha:
                nonsig.append((c, b, p))
        if not wrong and not nonsig:
            return res, features, history
        if wrong:
            wrong.sort(key=lambda r: abs(r[1]))
            drop = wrong[0][0]
            reason = 'wrong_sign'
        else:
            nonsig.sort(key=lambda r: -r[2])
            drop = nonsig[0][0]
            reason = f'p={nonsig[0][2]:.3g}'
        history.append((drop, reason))
        features.remove(drop)

    if len(features) < min_features:
        raise RuntimeError(
            f"Logit-2: po iteratívnom refite ostalo menej ako {min_features} premenných. "
            "Model nepoužívaj ako finálny benchmark bez úpravy kandidátnych premenných."
        )
    X = _design_matrix(X_train, features, step_name='Logit-2')
    res = _fit_robust(y, X)
    validate_logit_result(res, 'Logit-2 final', features, max_nan_p_share=0.0, require_converged=True)
    return res, features, history

print("Fit funkcie pripravené.")

## 6. Metrics and diagnostic tests

- `youden_threshold()` — optimal ROC-curve threshold from validation.
- `metrics_dict()` — ROC-AUC, PR-AUC, KS statistic, Brier score, balanced
  accuracy, recall and precision at the chosen threshold.
- `hosmer_lemeshow()` — decile-based goodness-of-fit; higher p-values
  indicate weaker evidence against calibration.
- `calibration_slope_intercept()` — recalibration logit on the predicted
  log-odds (ideal: intercept 0, slope 1).
- `brier_skill_score()` — BSS against a constant base-rate forecast.
- `topk_capture_lift()` — capture rate and lift in the top 1 / 5 / 10 % of
  the ranked predictions.
- `standardized_odds_ratios()` — odds ratio per one in-sample standard
  deviation of the predictor.
- `delong_test()` — paired AUC comparison helper. Not used in the test split
  (2018 vs 2019 are different firms); the notebook only reports per-year
  AUCs.

In [ ]:
def youden_threshold(probs, y):
    fpr, tpr, thr = roc_curve(y, probs); j = tpr - fpr
    i = int(np.nanargmax(j)); t = float(thr[i])
    return t if np.isfinite(t) else 0.5

def metrics_dict(y, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = (confusion_matrix(y, preds).ravel()
                      if len(np.unique(y)) > 1 else (0,0,0,0))
    ks = 0.0
    try:
        fpr, tpr, _ = roc_curve(y, probs)
        ks = float(np.max(np.abs(tpr - fpr)))
    except Exception: pass
    return {
        'AUC': float(roc_auc_score(y, probs)),
        'PR_AUC': float(average_precision_score(y, probs)),
        'KS': ks,
        'Accuracy': float(accuracy_score(y, preds)),
        'Brier': float(brier_score_loss(y, probs)),
        'Balanced_Accuracy': float(balanced_accuracy_score(y, preds)),
        'threshold': float(threshold),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'recall': float(tp/(tp+fn)) if (tp+fn) > 0 else 0.0,
        'precision': float(tp/(tp+fp)) if (tp+fp) > 0 else 0.0,
    }

def hosmer_lemeshow(y, probs, n_groups=10):
    df = pd.DataFrame({'y': y, 'p': probs}).reset_index(drop=True)
    df['decile'] = pd.qcut(df['p'], q=n_groups, duplicates='drop', labels=False)
    grp = df.groupby('decile').agg(observed=('y','sum'), expected=('p','sum'),
                                    n=('y','count')).reset_index()
    grp['n_obs0']    = grp['n'] - grp['observed']
    grp['expected0'] = grp['n'] - grp['expected']
    eps = 1e-12
    chi2 = float(((grp['observed']-grp['expected'])**2/(grp['expected']+eps) +
                  (grp['n_obs0']-grp['expected0'])**2/(grp['expected0']+eps)).sum())
    df_hl = max(len(grp)-2, 1)
    pval = float(1 - scistats.chi2.cdf(chi2, df_hl))
    return {'chi2': chi2, 'df': df_hl, 'p_value': pval, 'n_groups': int(len(grp))}

def calibration_slope_intercept(y, probs, eps=1e-9):
    p = np.clip(probs, eps, 1-eps); lp = np.log(p/(1-p))
    X = sm.add_constant(lp)
    try:
        res = sm.Logit(y, X).fit(disp=False, maxiter=500)
        return {'intercept': float(res.params[0]), 'slope': float(res.params[1])}
    except Exception:
        return {'intercept': np.nan, 'slope': np.nan}

def brier_skill_score(y, probs):
    base_rate = float(np.mean(y))
    bs_model  = float(brier_score_loss(y, probs))
    bs_naive  = float(brier_score_loss(y, np.full_like(y, base_rate, dtype=float)))
    return {'base_rate': base_rate, 'BS_model': bs_model, 'BS_naive': bs_naive,
            'BSS': float(1 - bs_model/bs_naive) if bs_naive > 0 else np.nan}

def topk_capture_lift(y, probs, ks=(0.01, 0.05, 0.10)):
    df = pd.DataFrame({'y': y, 'p': probs}).sort_values('p', ascending=False).reset_index(drop=True)
    n = len(df); pos_total = int(df['y'].sum())
    rows = []
    for k in ks:
        cut = max(int(np.ceil(n*k)), 1)
        cap = int(df['y'].iloc[:cut].sum())
        capture = cap/pos_total if pos_total > 0 else 0.0
        lift = (cap/cut)/(pos_total/n) if pos_total > 0 else 0.0
        rows.append({'k': k, 'cut': cut, 'captured': cap,
                     'capture_rate': capture, 'lift': lift})
    return rows

def standardized_odds_ratios(model, X_train, features):
    sd = X_train[features].std()
    rows = []
    for f in features:
        if f not in model.params.index: continue
        b = float(model.params[f])
        s = float(sd[f]) if not np.isnan(sd[f]) else 0.0
        rows.append({'variable': f, 'beta': b, 'sd': s,
                     'beta_x_sd': b*s, 'OR_per_1SD': float(np.exp(b*s))})
    out = pd.DataFrame(rows)
    if 'beta_x_sd' in out.columns:
        out['abs_bxs'] = out['beta_x_sd'].abs()
        out = out.sort_values('abs_bxs', ascending=False).drop(columns='abs_bxs')
    return out

def delong_test(y, probs1, probs2):
    y = np.asarray(y); p1 = np.asarray(probs1); p2 = np.asarray(probs2)
    pos = y==1; neg = y==0
    n1, n0 = pos.sum(), neg.sum()
    if n1 < 2 or n0 < 2:
        return {'AUC1': np.nan, 'AUC2': np.nan, 'z': np.nan, 'p_value': np.nan}
    def auc_components(probs, y):
        pos_p = probs[y==1]; neg_p = probs[y==0]
        V10 = np.array([np.mean((neg_p < x) + 0.5*(neg_p == x)) for x in pos_p])
        V01 = np.array([np.mean((pos_p > x) + 0.5*(pos_p == x)) for x in neg_p])
        return float(V10.mean()), V10, V01
    a1, V10_1, V01_1 = auc_components(p1, y)
    a2, V10_2, V01_2 = auc_components(p2, y)
    s10 = np.cov(np.stack([V10_1, V10_2]), bias=False) / n1
    s01 = np.cov(np.stack([V01_1, V01_2]), bias=False) / n0
    S = s10 + s01
    L = np.array([1, -1])
    var = float(L @ S @ L.T)
    if var <= 0:
        return {'AUC1': a1, 'AUC2': a2, 'z': np.nan, 'p_value': np.nan}
    z = (a1 - a2)/np.sqrt(var)
    pval = 2*(1 - scistats.norm.cdf(abs(z)))
    return {'AUC1': a1, 'AUC2': a2, 'z': float(z), 'p_value': float(pval)}

print("Diagnostické funkcie pripravené.")

## 7. Load data and build the variable set

Reads `data_logit_strict.csv`, calls `compute_ratios()` to ensure that required engineered ratio columns exist, builds year fixed effects with 2013 as reference, and dynamically detects the available `industry_*` dummies. The most frequent industry over the training period 2013–2016 is dropped as the reference category, leaving the remaining industries plus year FEs and base features as the candidate set `FEATURES`.

In [ ]:
print("Načítavanie ...")
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)
df.columns = df.columns.str.strip()
print(f"  shape: {df.shape}  | target rate: {df['target'].mean()*100:.3f}%")

df = compute_ratios(df)
df['rok'] = pd.to_numeric(df['rok'], errors='coerce').astype('Int64')

# Year FE: referenčný rok v tréningovom období = 2013.
for y in [2014, 2015, 2016]:
    df[f'year_{y}'] = (df['rok'] == y).astype(int)

# Industry features: načítanie všetkých dostupných industry_* dummy premenných.
industry_features_all = sorted([
    c for c in df.columns
    if c.startswith('industry_') and c not in {'industry_group', 'industry_slug'}
])
for c in industry_features_all:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(float)

# Vynechanie jednej referenčnej industry kategórie.
# Referencia sa volí podľa najpočetnejšej industry dummy v tréningovom období 2013–2016.
reference_industry = None
industry_features = industry_features_all.copy()
if len(industry_features_all) > 1:
    train_ref_mask = df['rok'].between(2013, 2016).fillna(False)
    ref_counts = df.loc[train_ref_mask, industry_features_all].sum(axis=0).sort_values(ascending=False)
    reference_industry = str(ref_counts.index[0])
    industry_features = [c for c in industry_features_all if c != reference_industry]
    print(f"  industry dummies celkovo: {len(industry_features_all)}")
    print(f"  referenčná industry kategória vynechaná z modelu: {reference_industry}")
    print(f"  industry dummies v modeli: {len(industry_features)}")
elif len(industry_features_all) == 1:
    print("  dostupná je iba jedna industry dummy premenná; referenčná kategória sa nevynecháva automaticky")
else:
    print("  industry dummies: 0")

FEATURES = list(dict.fromkeys(base_features + industry_features + year_features))
FEATURES = [f for f in FEATURES if f in df.columns]
print(f"  vstupných premenných: {len(FEATURES)}")

## 8. Time split, missingness check, training-only imputation

Splits into train (2013–2016) / valid (2017) / test (2018–2019). Variables with more than 5 % missing in the **training set** are dropped; variables with no variability in the training set are also dropped. Remaining `NaN` values are filled with **training-period medians** and the same medians are applied to validation and test. A final assertion ensures no NaN survives in the modeled feature matrices.

In [ ]:
train = df[df['rok'].between(2013, 2016)].copy()
valid = df[df['rok'] == 2017].copy()
test  = df[df['rok'].between(2018, 2019)].copy()

for d_ in (train, valid, test):
    for f in FEATURES:
        d_[f] = pd.to_numeric(d_[f], errors='coerce')
    d_[FEATURES] = d_[FEATURES].replace([np.inf, -np.inf], np.nan)

print(f"  Train: {len(train):>8,} | bk: {int(train['target'].sum())}")
print(f"  Valid: {len(valid):>8,} | bk: {int(valid['target'].sum())}")
print(f"  Test:  {len(test):>8,} | bk: {int(test['target'].sum())}")

# Filter missing ≤5 % sa musí počítať pred imputáciou.
miss = train[FEATURES].isna().mean()
keep_after_missing = [f for f in FEATURES if miss[f] <= 0.05]
dropped_missing = [f for f in FEATURES if miss[f] > 0.05]
print(f"  Po missing≤5%: {len(keep_after_missing)} (vyradených: {len(dropped_missing)})")
if dropped_missing:
    print("  Vyradené pre missing>5%:", dropped_missing[:20], "..." if len(dropped_missing) > 20 else "")

# Vyradenie premenných bez variability v tréningu.
nunique_train = train[keep_after_missing].nunique(dropna=False)
zero_var_features = [f for f in keep_after_missing if nunique_train[f] <= 1]
if zero_var_features:
    keep_after_missing = [f for f in keep_after_missing if f not in zero_var_features]
    print(f"  Vyradené pre nulovú variabilitu v tréningu: {zero_var_features}")

# Imputácia z train mediánov — fit na train, aplikácia na train/valid/test.
train_med = train[keep_after_missing].median(numeric_only=True)
for d_ in (train, valid, test):
    d_[keep_after_missing] = d_[keep_after_missing].fillna(train_med)

# Finálna poistka: po imputácii nesmú ostať NaN vo vybraných premenných.
for name, d_ in [('train', train), ('valid', valid), ('test', test)]:
    n_nan = int(d_[keep_after_missing].isna().sum().sum())
    if n_nan > 0:
        raise RuntimeError(f"Po imputácii ostali NaN hodnoty v množine {name}: {n_nan}")

y_train = train['target'].astype(int)
y_valid = valid['target'].astype(int)
y_test  = test['target'].astype(int)

## 9. VIF + full-rank filter + GVIF diagnostic

Applies the VIF filter to the non-protected continuous variables (threshold 10), then runs the full-rank filter to remove any variable that is an exact linear combination of the constant and the already-kept variables. Reports the number of features kept by group (continuous tested, protected continuous, dummy) and the GVIF for the industry block.

In [ ]:
print("VIF filter (threshold=10) — testujú sa iba nechránené spojité premenné")
feats_after_vif_before_rank = filter_vif(train, keep_after_missing, threshold=10.0)

print("\nFull-rank filter — odstránenie presne redundantných premenných pred Logit-1")
feats_vif, dropped_full_rank = make_full_rank_features(train, feats_after_vif_before_rank, verbose=True)

n_dummy_kept = sum(1 for f in feats_vif if is_dummy(f))
n_protected_cont_kept = sum(1 for f in feats_vif if f in PROTECTED_FEATURES)
n_cont_vif_kept = sum(1 for f in feats_vif if not is_protected(f))
n_industry_kept = sum(1 for f in feats_vif if f.startswith('industry_'))
n_year_kept = sum(1 for f in feats_vif if f.startswith('year_'))

print(f"  Po VIF + full-rank: {len(feats_vif)} prem. "
      f"(testované spojité={n_cont_vif_kept}, protected_cont={n_protected_cont_kept}, dummy={n_dummy_kept})")
print(f"  Industry kept: {n_industry_kept}/{len(industry_features)} "
      f"(referencia: {reference_industry})")
print(f"  Year kept: {n_year_kept}/3")

vif_table = compute_vif(train, [f for f in feats_vif if not is_protected(f)])
print("\nTop 8 VIF (testované spojité):")
print(vif_table.head(8).to_string(index=False) if len(vif_table) else "  Žiadne spojité premenné na VIF test.")

# GVIF pre industry blok
print("\n[GVIF] Industry blok (Fox & Monette 1992):")
ind_in = [f for f in feats_vif if f.startswith('industry_')]
other_for_gvif = [f for f in feats_vif if not f.startswith('industry_')]
gvif = compute_gvif_block(train, ind_in, other_for_gvif) if ind_in else None
if gvif:
    gn = gvif['GVIF_normalized']
    gn_str = "NaN" if (gn is None or np.isnan(gn)) else f"{gn:.3f}"
    print(f"  n_block: {gvif['n_block']}")
    print(f"  Max VIF v bloku: {gvif['max_vif_within_block']:.2f}")
    print(f"  GVIF^(1/(2df)): {gn_str}")
    print(f"  Verdikt: {'OK (<3.16)' if (gn is not None and not np.isnan(gn) and gn < 3.16) else 'WATCH'}")
else:
    print("  Industry blok nie je dostupný alebo bol prázdny.")


## 10. Logit-1 screening

Fits the screening logit on the post-filter feature set. The result is validated on convergence and on the absence of NaN p-values; the notebook aborts otherwise. `select_from_logit1` then keeps only variables that pass both the sign prior and the significance test (initially α = 0.05, relaxed to 0.10 only if fewer than three variables would survive).

In [ ]:
print(f"Logit-1 screening na {len(train):,} obs × {len(feats_vif)} prem.")
t1 = time.time()
l1 = fit_logit1(train, y_train, feats_vif)
print(f"  fit: {time.time()-t1:.1f}s | converged={l1.mle_retvals.get('converged','?')}")

tbl_l1 = validate_logit_result(l1, 'Logit-1', feats_vif, max_nan_p_share=0.0, require_converged=True)

feats_l2, dropped_l2 = select_from_logit1(l1, feats_vif, EXPECTED_SIGNS, alpha=0.05)
if len(feats_l2) < 3:
    print("  alpha=0.05 ponechala menej ako 3 premenné, relaxujem na 0.10 ...")
    feats_l2, dropped_l2 = select_from_logit1(l1, feats_vif, EXPECTED_SIGNS, alpha=0.10)

if len(feats_l2) < 3:
    raise RuntimeError(
        "Logit-1 ponechal menej ako 3 kandidátne premenné aj pri alpha=0.10. "
        "Nepokračujem fallback výberom; skontroluj kandidátne premenné, separáciu alebo kolinearitu."
    )

drop_reasons = pd.Series([r for _, r in dropped_l2]).value_counts().head(10)
print(f"  Kandidáti pre Logit-2: {len(feats_l2)}")
print(f"  {feats_l2}")
print("\n  Najčastejšie dôvody vyradenia v Logit-1:")
print(drop_reasons.to_string() if len(drop_reasons) else "  Žiadne vyradené premenné.")

## 11. Logit-2 — iterative refit on selected variables

Starts from the Logit-1 candidates and iteratively drops the single worst remaining wrong-sign variable (preferred) or, failing that, the least significant variable. Each intermediate fit is validated. The loop stops once all coefficients have the expected sign and a p-value below α; the notebook errors out if fewer than three variables would remain.

In [ ]:
print("Logit-2 iteratívny refit ...")
t2 = time.time()
l2, feats_final, history = fit_logit2_iterative(
    train, y_train, feats_l2, EXPECTED_SIGNS, alpha=0.05, min_features=3
)
print(f"  refit: {time.time()-t2:.1f}s")
tbl_l2 = validate_logit_result(l2, 'Logit-2 final', feats_final, max_nan_p_share=0.0, require_converged=True)

print(f"\n  FINÁLNE PREMENNÉ ({len(feats_final)}): {feats_final}")
if history:
    print("  Vyradené počas refitu:")
    for d, r in history:
        print(f"    - {d}: {r}")
else:
    print("  Počas refitu nebolo potrebné vyradiť ďalšie premenné.")

## 12. Validation and test evaluation

Predicts probabilities on the validation and test splits, picks the Youden-J threshold from validation, and reports ROC-AUC, PR-AUC, KS, Brier, recall and precision on both splits.

In [ ]:
Xv = sm.add_constant(valid[feats_final].fillna(0), has_constant='add')
Xt = sm.add_constant(test[feats_final].fillna(0), has_constant='add')
pv = l2.predict(Xv); pt = l2.predict(Xt)
thr = youden_threshold(pv, y_valid)

m_valid = metrics_dict(y_valid, pv, thr)
m_test  = metrics_dict(y_test,  pt, thr)

print("METRIKY:")
print(f"  Valid: AUC={m_valid['AUC']:.4f} | PR-AUC={m_valid['PR_AUC']:.4f} | "
      f"KS={m_valid['KS']:.4f} | Brier={m_valid['Brier']:.6f}")
print(f"  Test:  AUC={m_test['AUC']:.4f}  | PR-AUC={m_test['PR_AUC']:.4f}  | "
      f"KS={m_test['KS']:.4f}  | Brier={m_test['Brier']:.6f}")
print(f"  Test recall: {m_test['recall']*100:.1f}% | precision: {m_test['precision']*100:.2f}% "
      f"(thr={thr:.4f})")

## 13. Test-set diagnostics

Hosmer–Lemeshow goodness-of-fit on deciles, calibration slope and intercept, Brier Skill Score against the base rate, top-k capture and lift at 1 % / 5 % / 10 %, and the top-10 standardised odds ratios (per 1 SD).

In [ ]:
# Hosmer-Lemeshow
hl = hosmer_lemeshow(y_test.values, pt.values, n_groups=10)
hl_verdict = "kalibrácia OK" if hl['p_value'] > 0.05 else "POSSIBLE MISCALIBRATION"
print(f"  Hosmer–Lemeshow: chi2={hl['chi2']:.2f} | df={hl['df']} | p={hl['p_value']:.4g}  ({hl_verdict})")

# Calibration
cs = calibration_slope_intercept(y_test.values, pt.values)
print(f"  Calibration: intercept={cs['intercept']:.3f} | slope={cs['slope']:.3f}  (ideál 0, 1)")

# BSS
bss = brier_skill_score(y_test.values, pt.values)
print(f"  Brier Skill Score: BSS={bss['BSS']:+.4f}  "
      f"(BS_model={bss['BS_model']:.6f} vs naive={bss['BS_naive']:.6f})")

# Top-k
topk = topk_capture_lift(y_test.values, pt.values)
print(f"\n  Top-k capture & lift:")
for r in topk:
    print(f"    top {int(r['k']*100):>3}%: cap={r['captured']:>4d}/{int(y_test.sum())} "
          f"({r['capture_rate']*100:5.1f}%) | lift={r['lift']:.2f}x")

# Standardized OR
std_or = standardized_odds_ratios(l2, train, feats_final)
print(f"\n  Top 10 štandardizovaných odds ratios (per 1 SD):")
print(std_or.head(10).to_string(index=False))

## 14. Per-year AUC decomposition (2018 vs 2019)

Reports test-set ROC-AUC separately for 2018 and 2019. A paired DeLong test is **not** computed because the two years contain different firms; only the marginal AUCs and their difference are reported.

In [ ]:
test18 = test[test['rok'] == 2018].copy()
test19 = test[test['rok'] == 2019].copy()
delong = None
if len(test18) > 0 and len(test19) > 0:
    X18 = sm.add_constant(test18[feats_final].fillna(0), has_constant='add')
    X19 = sm.add_constant(test19[feats_final].fillna(0), has_constant='add')
    p18 = l2.predict(X18); p19 = l2.predict(X19)
    try:
        auc18 = roc_auc_score(test18['target'], p18)
        auc19 = roc_auc_score(test19['target'], p19)
    except Exception:
        auc18 = auc19 = np.nan
    delong = {'AUC_2018': float(auc18), 'AUC_2019': float(auc19),
              'note': 'Paired DeLong nie je vhodný (rôzne firmy v rokoch)'}
    print(f"  AUC 2018: {auc18:.4f}")
    print(f"  AUC 2019: {auc19:.4f}")
    print(f"  Δ AUC:    {auc18-auc19:+.4f}")

## 15. Save predictions and summary

Writes `logit_v7_results.pkl` (validation and test predictions, ids, target, final feature list, threshold, full metrics) for the downstream stacking notebook, and `logit_v7_results.json` with the metrics, the dropped-variable history at each filtering stage, and the GVIF / Hosmer–Lemeshow / calibration / BSS / top-k / standardised-OR diagnostics.

In [ ]:
# Pickle pre stacking/model comparison.
stacking_export = {
    'model': 'logit2_strict_benchmark',
    'version': 'v7_fixed_fullrank',
    'industry_reference_dropped': reference_industry,

    'id_valid':  valid['id'].values,
    'rok_valid': valid['rok'].values.astype(int),
    'y_valid':   y_valid.values,
    'probs_valid': pv.values,

    'id_test':  test['id'].values,
    'rok_test': test['rok'].values.astype(int),
    'y_test':   y_test.values,
    'probs_test': pt.values,

    'features_after_vif_before_full_rank': feats_after_vif_before_rank,
    'features_dropped_full_rank': dropped_full_rank,
    'features_logit1_after_vif_full_rank': feats_vif,
    'features_logit2_candidates': feats_l2,
    'features_logit2': feats_final,
    'threshold_youden_valid': float(thr),
    'metrics_valid': m_valid,
    'metrics_test':  m_test,
}
with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(stacking_export, f)
print(f"Stacking pickle: {RESULTS_PATH}")

# JSON sumár.
summary = {
    'model': 'logit2_strict_benchmark',
    'version': 'v7_fixed_fullrank',
    'data_shape': list(df.shape),
    'target_rate': float(df['target'].mean()),
    'split': {'train': '2013-2016', 'valid': '2017', 'test': '2018-2019'},

    'features_input': len(FEATURES),
    'features_after_missing_and_zero_variance': len(keep_after_missing),
    'dropped_missing_gt_5pct': dropped_missing,
    'dropped_zero_variance': zero_var_features,
    'features_after_vif_before_full_rank_count': len(feats_after_vif_before_rank),
    'features_dropped_full_rank_count': len(dropped_full_rank),
    'features_after_vif_full_rank_count': len(feats_vif),

    'dummy_features_kept': n_dummy_kept,
    'protected_continuous_kept': n_protected_cont_kept,
    'continuous_vif_tested_kept': n_cont_vif_kept,
    'industry_reference_dropped': reference_industry,
    'industry_total_before_reference': len(industry_features_all),
    'industry_in_model_after_reference': len(industry_features),
    'industry_kept_after_vif': n_industry_kept,
    'year_kept': n_year_kept,

    'features_after_vif_before_full_rank': feats_after_vif_before_rank,
    'features_dropped_full_rank': dropped_full_rank,
    'features_logit1_after_vif_full_rank': feats_vif,
    'features_logit2_candidates': feats_l2,
    'logit1_dropped': dropped_l2,
    'features_logit2_final': feats_final,
    'logit2_history': history,

    'metrics_valid': m_valid,
    'metrics_test':  m_test,
    'gvif_industry_block': gvif,
    'hosmer_lemeshow': hl,
    'calibration': cs,
    'brier_skill_score': bss,
    'topk_lift': topk,
    'standardized_or_top10': std_or.head(10).to_dict(orient='records'),
    'auc_year_compare': delong,
}
with open(JSON_PATH, 'w') as f:
    json.dump(summary, f, default=str, indent=2)
print(f"JSON summary: {JSON_PATH}")

print("\n========== HOTOVO ==========")
print(f"Logit-2 final: {len(feats_final)} premenných")
print(f"Test: AUC={m_test['AUC']:.4f} | PR-AUC={m_test['PR_AUC']:.4f} | KS={m_test['KS']:.4f}")